# FreePhish — Differential Evolution (DE) feature-weight search
Runs `freephish_de.py`, which DE-searches a per-feature weight vector for the **enhanced (100-feature) model only**, then reports accuracy/precision/recall/F1 on a held-out test set.

**Cost warning:** DE retrains a full GBDT+XGBoost+LightGBM stack for every individual, every generation — roughly `pop-size * (generations + 1)` full fits. Run the small smoke-test cell first to see your per-fit time on your real data, then size `--pop-size`/`--generations` accordingly before committing to a long run.

In [ ]:
%cd /content
!rm -f freephish_*.py requirements.txt
from google.colab import files
files.upload()   # select: freephish_old.py, freephish_enhanced.py, freephish_de.py, requirements.txt
!ls

In [ ]:
!pip install -q -r requirements.txt

## Dataset
Download a balanced sample, or skip this cell and point `--csv` at your own file below if you already have one (e.g. the `sample.parquet` from the main pipeline notebook).

In [ ]:
!python freephish_old.py download --repo phreshphish/phreshphish --max-shards 20 --per-class 3118 --out sample.parquet

In [ ]:
import pyarrow.parquet as pq, pandas as pd
print(pq.ParquetFile("sample.parquet").metadata.num_rows, "rows")
print(pd.read_parquet("sample.parquet", columns=["url", "label"])["label"].value_counts())

## Step 1: smoke test (small pop-size/generations)
Confirms the pipeline runs end to end on your data and shows you the per-fit time, so you can size the real run sensibly. Safe to run first — only 4*(2+1) = 12 StackModel fits.

In [ ]:
import time
t0 = time.time()
!python freephish_de.py train --csv sample.parquet --pop-size 4 --generations 2 \
    --results-out results_de_smoke.json --model model_de_smoke.joblib
print(f"\nsmoke test wall-clock: {time.time() - t0:.0f}s for 12 fits -> ~{(time.time() - t0) / 12:.1f}s/fit")

## Step 2: full DE run
Defaults are `--pop-size 10 --generations 10` (110 fits) — adjust based on the per-fit time you just saw. `--split group` (default) keeps test-set domains unseen during training; use `--split random` to match the paper's 70/30 random split instead.

In [ ]:
!python freephish_de.py train --csv sample.parquet \
    --pop-size 10 --generations 10 --F 0.5 --CR 0.7 \
    --results-out results_de.json --model model_de.joblib

### Read back accuracy / precision / recall / F1

In [ ]:
import json
m = json.load(open("results_de.json"))["metrics"]
print(f"accuracy  {m['accuracy']:.4f}")
print(f"precision {m['precision']:.4f}")
print(f"recall    {m['recall']:.4f}")
print(f"f1        {m['f1']:.4f}")

### Compare against the plain enhanced model (optional)
Requires `results_enhanced.json` from the main pipeline notebook (Stage 2) to already exist in this session.

In [ ]:
import json, os
if os.path.exists("results_enhanced.json"):
    base = json.load(open("results_enhanced.json"))["metrics"]
    de = json.load(open("results_de.json"))["metrics"]
    print(f"{'metric':<12}{'enhanced':>12}{'enhanced+DE':>14}{'difference':>14}")
    for k in ("accuracy", "precision", "recall", "f1"):
        print(f"{k:<12}{base[k]:>12.4f}{de[k]:>14.4f}{de[k] - base[k]:>+14.4f}")
else:
    print("results_enhanced.json not found in this session — run freephish_enhanced.py train first (see the main pipeline notebook) to compare.")

### Download results

In [ ]:
from google.colab import files
import os
for f in ["results_de.json", "results_de.predictions.csv", "model_de.joblib"]:
    if os.path.exists(f):
        files.download(f)